# Traffic Signal RL - SUMO Training (Google Colab)

Train a PPO agent to control traffic signals using SUMO scenarios.

**Runtime:** No GPU needed (CPU is fine for this).

## 1. Install Dependencies

In [ ]:
!pip install eclipse-sumo traci sumolib sumo-rl stable-baselines3 torch tensorboard -q

In [ ]:
import os
import sumo
os.environ["SUMO_HOME"] = os.path.dirname(sumo.__file__)
print(f"SUMO_HOME set to: {os.environ['SUMO_HOME']}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sumo_rl
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import EvalCallback

NETS_DIR = os.path.join(os.path.dirname(sumo_rl.__file__), "nets")
print("All imports OK")
print(f"Available scenarios: {os.listdir(NETS_DIR)}")

## 2. Create SUMO Environment

Choose a scenario by changing `SCENARIO` below:
- `"single-intersection"` - 4-way intersection (simplest)
- `"2way-single-intersection"` - 2-way intersection
- `"RESCO/cologne1"` - Real-world Cologne intersection
- `"RESCO/ingolstadt1"` - Real-world Ingolstadt intersection
- `"RESCO/grid4x4"` - 4x4 grid (advanced)

In [ ]:
# === CHANGE SCENARIO HERE ===
SCENARIO = "single-intersection"  # try "RESCO/cologne1" for real-world data
TOTAL_TIMESTEPS = 50_000          # increase to 200k+ for better results
# ============================

# Find the net and route files
scenario_dir = os.path.join(NETS_DIR, SCENARIO)
net_file = [f for f in os.listdir(scenario_dir) if f.endswith(".net.xml")][0]
rou_file = [f for f in os.listdir(scenario_dir) if f.endswith(".rou.xml")][0]

print(f"Scenario: {SCENARIO}")
print(f"Net file: {net_file}")
print(f"Route file: {rou_file}")

In [ ]:
def make_env():
    return sumo_rl.SumoEnvironment(
        net_file=os.path.join(scenario_dir, net_file),
        route_file=os.path.join(scenario_dir, rou_file),
        use_gui=False,
        num_seconds=5000,
        delta_time=5,
        yellow_time=3,
        min_green=10,
        max_green=50,
        single_agent=True,
    )

# Quick test
env = make_env()
obs, info = env.reset()
print(f"Observation space: {env.observation_space}")
print(f"Action space: {env.action_space}")
print(f"Initial obs: {obs}")
env.close()
print("Environment test PASSED")

## 3. Train PPO Agent

In [ ]:
train_env = make_env()

model = PPO(
    "MlpPolicy",
    train_env,
    verbose=1,
    learning_rate=3e-4,
    n_steps=2048,
    batch_size=64,
    n_epochs=10,
    gamma=0.99,
    ent_coef=0.01,
)

print(f"Training for {TOTAL_TIMESTEPS:,} timesteps...")
model.learn(total_timesteps=TOTAL_TIMESTEPS)
print("Training complete!")

train_env.close()

## 4. Evaluate Trained Agent vs Random Baseline

In [ ]:
def evaluate(model, env, n_episodes=5, name="Agent"):
    """Evaluate a model and return episode rewards."""
    rewards = []
    for ep in range(n_episodes):
        obs, info = env.reset()
        total_reward = 0
        done = False
        while not done:
            if model is None:  # random baseline
                action = env.action_space.sample()
            else:
                action, _ = model.predict(obs, deterministic=True)
            obs, reward, terminated, truncated, info = env.step(action)
            total_reward += reward
            done = terminated or truncated
        rewards.append(total_reward)
        print(f"  {name} Episode {ep+1}: reward = {total_reward:.2f}")
    return rewards

eval_env = make_env()

print("=== Trained PPO Agent ===")
ppo_rewards = evaluate(model, eval_env, n_episodes=3, name="PPO")

print("\n=== Random Baseline ===")
random_rewards = evaluate(None, eval_env, n_episodes=3, name="Random")

eval_env.close()

print(f"\n--- Results ---")
print(f"PPO avg reward:    {np.mean(ppo_rewards):.2f} +/- {np.std(ppo_rewards):.2f}")
print(f"Random avg reward: {np.mean(random_rewards):.2f} +/- {np.std(random_rewards):.2f}")
print(f"Improvement: {((np.mean(ppo_rewards) - np.mean(random_rewards)) / abs(np.mean(random_rewards)) * 100):.1f}%")

## 5. Visualize Results

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

x = np.arange(len(ppo_rewards))
width = 0.35

ax.bar(x - width/2, ppo_rewards, width, label='PPO Agent', color='#2ecc71')
ax.bar(x + width/2, random_rewards, width, label='Random', color='#e74c3c')

ax.set_xlabel('Episode')
ax.set_ylabel('Total Reward')
ax.set_title(f'PPO vs Random - {SCENARIO}')
ax.set_xticks(x)
ax.set_xticklabels([f'Ep {i+1}' for i in range(len(ppo_rewards))])
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nPPO outperforms random by {np.mean(ppo_rewards) - np.mean(random_rewards):.2f} reward points")

## 6. Save Model (optional)

In [ ]:
model.save("ppo_traffic_signal")
print("Model saved to ppo_traffic_signal.zip")

# Download from Colab:
# from google.colab import files
# files.download("ppo_traffic_signal.zip")